# Re-probe the LoRA adapters on What'sUp

Runtime → Change runtime type → **T4 GPU**. Needs the `GITHUB_TOKEN` secret, as in the training notebook.

For each adapter in `RUNS`: `src.extract_hidden` saves What'sUp hidden states with the adapter loaded
(about 5 min on the GPU), then `src.probe` fits the same per-layer probes as for the base model
(about 7 min on the CPU). Results go to `spatial_probe/results/` next to the base model's
(`hidden/`, `probes/`, `figures/`, `metrics/`, `preds/`, all prefixed with the run name).

`extract_hidden` prints a warning when depth accuracy is at least 85%. That rule was for the base
model, so ignore it here.

If Colab disconnects, run all cells again: finished steps are skipped and extraction resumes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from google.colab import userdata

BRANCH = 'main'
REPO = '/content/spatial-probing'

token = userdata.get('GITHUB_TOKEN')
if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} https://{token}@github.com/vitorizz/spatial-probing.git {REPO}
%cd {REPO}
!git checkout -q {BRANCH} && git pull -q
!git log --oneline -1

In [ ]:
# Same setup as the training notebook: keep Colab's CUDA torch, drop torchao.
!grep -v -E '^torch' requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt
!pip uninstall -y -q torchao
!pip list 2>/dev/null | grep -E '^(torch|transformers|peft|accelerate) '
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
RUNS = ['lora_targeted', 'lora_all', 'lora_random']

import subprocess, sys
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/spatial_probe')
RESULTS = DRIVE_ROOT / 'results'

for run in RUNS:
    adapter = DRIVE_ROOT / 'runs' / run / 'adapter'
    assert (adapter / 'adapter_model.safetensors').is_file(), f'no finished adapter at {adapter}'
    if not (RESULTS / 'metrics' / f'{run}_whatsup.json').exists():
        subprocess.run([sys.executable, '-m', 'src.extract_hidden', '--adapter', str(adapter), '--model-name', run], check=True)
    if not (RESULTS / 'probes' / f'{run}_probe_summary.json').exists():
        subprocess.run([sys.executable, '-m', 'src.probe', '--model-name', run], check=True)

In [ ]:
import json
for name in ['base', *RUNS]:
    s = json.loads((RESULTS / 'probes' / f'{name}_probe_summary.json').read_text())
    m = json.loads((RESULTS / 'metrics' / f'{name}_whatsup.json').read_text())
    print(f"{name:14} model depth acc {m['depth_accuracy']:.3f} | final-layer probe {s['final_layer_acc']}"
          f" | first layer >= 0.95 {s['first_layer_acc_ge_0.95']} | shuffled pass {s['shuffled_control']['pass_within_5_points']}")